[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-05-sentence-alignment.ipynb)


# CS 479, Lecture 5: In-Class Activity
## When the data doesn't come aligned

**Ungraded. Work in pairs. About 15 minutes.**

Your Church data arrived aligned, because translators built it one segment at a time.
Most parallel text in the world arrives as whole documents instead: a book and its
translation, a treaty, a website. Somebody has to line up the sentences.

Today you will run the Gale-Church aligner, find the one thing it gets wrong, try to fix it,
and then measure your own language pair the way Gale and Church measured theirs.

**Two numbers to know before you start.** The first, **c**, is the average number of characters in the
target language per character of English. If a 100-character English sentence typically becomes
a 110-character Spanish sentence, c is about 1.1: Spanish runs about 10% longer. Gale-Church
leans on c to decide which sentences belong together. A second number, **s²**, says how much
individual sentence pairs wobble around that ratio. NLTK's defaults are c = 1 and s² = 6.8.

**Before you start:** File > Save a copy in Drive.


---
## Step 1. Install

NLTK ships a maintained Gale-Church aligner. Upgrading makes sure you have the current one.


In [ ]:
!pip install -q -U nltk

## Step 2. A document and its translation

Thirteen English sentences, eleven Spanish ones. The translator did what translators do:
split one sentence, merged two pairs, and left one sentence out.

`GOLD` is the right answer, written by someone who reads both languages. The aligner never sees it.


In [ ]:
EN = [
    "Every translation memory begins as ordinary work.",
    "A translator opens a document, reads a sentence, and writes its translation.",
    "The tool saves the pair.",
    "Over years, those pairs pile up by the million.",
    "Most of the world's parallel text was never produced this way.",
    "It exists as whole documents: a book and its translation, a treaty in six languages, a website in two.",
    "Nobody lined up the sentences.",
    "That job falls to us.",
    "Fortunately, long sentences tend to have long translations.",
    "Short ones tend to have short ones.",
    "In 1993, William Gale and Kenneth Church turned that simple observation into an algorithm that is still in use today.",
    "It knows nothing about words.",
    "It counts characters.",
]
ES = [
    "Toda memoria de traducción comienza como trabajo ordinario.",
    "Un traductor abre un documento, lee una oración y escribe su traducción.",
    "La herramienta guarda el par.",
    "Con los años, esos pares se acumulan por millones.",
    "La mayor parte del texto paralelo del mundo nunca se produjo de esta manera.",
    "Existe como documentos completos: un libro y su traducción, un tratado en seis idiomas.",
    "Un sitio web en dos.",
    "Nadie alineó las oraciones; ese trabajo nos toca a nosotros.",
    "Por fortuna, las oraciones largas suelen tener traducciones largas, y las cortas, traducciones cortas.",
    "En 1993, William Gale y Kenneth Church convirtieron esa sencilla observación en un algoritmo que todavía se usa hoy.",
    "Cuenta caracteres.",
]

# The right answer, written by a person who can read both sides.
# English sentence index -> the Spanish sentence indices it belongs with.
GOLD = {0: [0], 1: [1], 2: [2], 3: [3], 4: [4], 5: [5, 6], 6: [7], 7: [7],
        8: [8], 9: [8], 10: [9], 11: [], 12: [10]}
GOLD_LINKS = {(e, s) for e, ss in GOLD.items() for s in ss}

print(len(EN), "English sentences,", len(ES), "Spanish sentences")


## Step 3. Align by length

`align_blocks` takes nothing but the **character length** of each sentence on each side.
No dictionary, no model, no idea what any word means.

Each output pair `(i, j)` links English sentence `i` to Spanish sentence `j`. A sentence that
appears twice is part of a merge or a split.


In [ ]:
from nltk.translate.gale_church import align_blocks, LanguageIndependent

def lengths(sents):
    return [len(s) for s in sents]          # characters, exactly as Gale and Church did

def score(links):
    links = set(links)
    hit = links & GOLD_LINKS
    print(f"precision {len(hit)}/{len(links)}   recall {len(hit)}/{len(GOLD_LINKS)}")
    if links - GOLD_LINKS: print("  wrong links :", sorted(links - GOLD_LINKS))
    if GOLD_LINKS - links: print("  missed links:", sorted(GOLD_LINKS - links))

def show(links):
    for e, s in links:
        mark = "  " if (e, s) in GOLD_LINKS else "✗ "
        print(f"{mark}EN{e:>2} ({len(EN[e]):>3})  {EN[e][:44]:<44}  ->  ES{s:>2} ({len(ES[s]):>3})  {ES[s][:40]}")

links = align_blocks(lengths(EN), lengths(ES))
show(links)
print()
score(links)


### What just happened

Everything is right except one link, marked `✗`. English sentence 11, "It knows nothing about
words.", has no Spanish translation. The aligner did not call it a deletion. It folded it into
the sentence before it, as a 2-1 merge.

**Before you run the next cell:** why do you think it preferred that?


## Step 4. Your turn: fix it

The priors say how common each kind of link is. Change them and see whether you can make the
aligner find the deletion without breaking the genuine split and merges.


In [ ]:
# TODO: can you get the aligner to find the dropped sentence?
# The priors say how common each kind of link is. Gale and Church measured them;
# a 1-0 link (a sentence with no translation) is rare, so it is expensive.
# Try raising the deletion prior, and lowering the merge prior, and re-run.

DELETION_PRIOR = 0.0099      # Gale and Church: 0.0099 each for 1-0 and 0-1
MERGE_PRIOR    = 0.089       # Gale and Church: 0.089 each for 2-1 and 1-2

class MyParams(LanguageIndependent):
    PRIORS = dict(LanguageIndependent.PRIORS)
    PRIORS[(1, 0)] = PRIORS[(0, 1)] = DELETION_PRIOR
    PRIORS[(2, 1)] = PRIORS[(1, 2)] = MERGE_PRIOR

score(align_blocks(lengths(EN), lengths(ES), MyParams))


### Why nothing works

It is not the priors. To an aligner that only sees lengths, "a short sentence with no translation"
and "a short sentence absorbed into its neighbour's translation" look nearly the same: 146
English characters against 116 Spanish is only mildly surprising. Lower the merge prior far
enough to forbid that, and you forbid the real merges too.

**Length cannot see a deletion. Meaning can.** Moore (2002) fixed this by adding word
translation probabilities, from IBM Model 1, which we meet later in the course. Modern aligners
such as Vecalign (Thompson and Koehn, 2019) compare multilingual sentence embeddings, the same
shared space you tested in Lecture 3.


## Step 5. Measure your own language pair

Gale and Church did not guess their parameters. They measured them on text that was already
aligned: **c**, how many target characters per source character, and **s2**, how much that
ratio wobbles. Your Assignment 4 files are aligned text. Measure yours.


In [ ]:
# Estimate c and s2 for YOUR language pair, the way Gale and Church did:
# from text that is already aligned. Your Assignment 4 files are exactly that.
#
# Upload them with the Files panel on the left (folder icon), then set the two paths.
# They stay in this runtime and disappear when it ends. Do not save them into a shared copy.

import os, statistics

EN_PATH = "english.txt"        # <-- your English file
TX_PATH = "target.txt"         # <-- your language's file
LANGUAGE = "my language"

if os.path.exists(EN_PATH) and os.path.exists(TX_PATH):
    en_lines = open(EN_PATH, encoding="utf-8").read().splitlines()
    tx_lines = open(TX_PATH, encoding="utf-8").read().splitlines()
    print(f"loaded {len(en_lines):,} English and {len(tx_lines):,} {LANGUAGE} lines")
else:
    print("Files not found, so using the 1-1 pairs from the sample instead.")
    one_to_one = [(e, s[0]) for e, s in GOLD.items() if len(s) == 1 and sum(s[0] in v for v in GOLD.values()) == 1]
    en_lines = [EN[e] for e, _ in one_to_one]
    tx_lines = [ES[s] for _, s in one_to_one]
    LANGUAGE = "Spanish (sample)"

pairs = [(len(a), len(b)) for a, b in zip(en_lines, tx_lines) if a.strip() and b.strip()]
c = sum(t for _, t in pairs) / sum(s for s, _ in pairs)
s2 = statistics.mean((c * s - t) ** 2 / ((s + t / c) / 2) for s, t in pairs)

print(f"\n{LANGUAGE}: c = {c:.2f}  (NLTK default 1.0)    s2 = {s2:.1f}  (NLTK default 6.8)")
print(f"{LANGUAGE} runs {abs(c - 1) * 100:.0f}% {'longer' if c > 1 else 'shorter'} than English, in characters.")


In [ ]:
# While you have your files open: if row n is a translation of row n, sentence lengths
# correlate strongly. Slip one side by a single row and the correlation should collapse.

import numpy as np

def length_correlation(a_lines, b_lines):
    a = np.array([len(x) for x in a_lines], dtype=float)
    b = np.array([len(x) for x in b_lines], dtype=float)
    return float(np.corrcoef(a, b)[0, 1])

n = min(len(en_lines), len(tx_lines))
print(f"as aligned        : {length_correlation(en_lines[:n], tx_lines[:n]):.3f}")
print(f"slipped by one row: {length_correlation(en_lines[:n-1], tx_lines[1:n]):.3f}")


## Step 6. Report back

We will build a table on the board.

1. **What is your c?** Which languages in the room run longer than English, which shorter, and by how much?
2. Is anyone's c far from 1? What would the default parameters do to their alignment?
3. What did you try in step 4, and what broke?
4. Your Church data came aligned. Name one source of parallel text in your language that would not.


---
### A reminder on AI use

Per Lecture 1: you may ask AI what a library function does. Do not have it write your algorithm
or your code, and be ready to explain every line you submit.

BYU CS AI policy: https://cs.byu.edu/department/ai-policy
